In [2]:
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from pydub import AudioSegment

# 1. Load data

In [3]:
bundle = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
model = bundle.get_model()

audio_file = "sample_audio.wav"
audio, sr = librosa.load(audio_file, sr=None)
clip = AudioSegment.from_wav(audio_file)

# 2. Export Function

In [4]:
def export_pydub(audio):
    while True:
        file_name = input("Input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must be ends with .wav")
    audio.export(file_name, format="wav")
    print("Audio Saved as: ", file_name)

def export_librosa(audio):
    while True:
        file_name = input("Input file name: ")
        if file_name.endswith(".wav"):
            break
        else:
            print("file must be ends with .wav")
    sf.write(file_name, np.clip(audio, -1, 1), sr)
    print("Audio Saved as: ", file_name)

# 3. Predict Text wav2vec

In [5]:
def predict_text(audio_file):
    audio_np, _ = librosa.load(audio_file, sr=bundle.sample_rate, mono=True)
    waveform = torch.from_numpy(audio_np).unsqueeze(0)

    with torch.inference_mode():
        emission, _ = model(waveform)

    class GreedyCTCDecoder(torch.nn.Module):
        def __init__(self, labels):
            super().__init__()
            self.labels = labels
            self.blank = 0

        def forward(self, emission: torch.Tensor)->str:
            indices = torch.argmax(emission, dim=-1)
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            return "".join([self.labels[i] for i in indices])

    decoder = GreedyCTCDecoder(bundle.get_labels())
    transcript = decoder(emission[0])
    transcript = transcript.replace("|", " ")
    return transcript

# 4. augmentation

In [ ]:
# augmentation ada 9
# 1. adjvolume
def adjvolume(clip):
    while True:
        nilai = float(input("Input nilai volume (1-5):"))
        if not (1 <= nilai <= 5):
            print("Nilai must be between 1-5.")
        else:
            break
    export_pydub(clip + nilai)

# 2. repeat audio
def repeat_audio(clip):
    while True:
        nilai = int(input("Input repeat audio must greater than 0:"))
        if nilai <=0:
            print("Nilai must be greater than 0.")
        else:
            break
    export_pydub(clip * nilai)

# 3. fade in
def fade_in(clip):
    while True:
        duration = int(input("Input duration fade in must greater than 0:"))
        if duration <=0:
            print("duration must be grater than 0.")
        else:
            break
    export_pydub(clip.fade_in(duration))

# import random
# def fade_in(clip):
#     nilai = random.randint(1, 5)
#     export_pydub(clip.fade_in(nilai * 1000))

# 4. fade out
def fade_out(clip):
    while True:
        duration = int(input("Input duration fade out must greater than 0:"))
        if duration <=0:
            print("duration must be grater than 0.")
        else:
            break
    export_pydub(clip.fade_out(duration))

# 5. reverse audio
def reverse_audio(clip):
    export_pydub(clip.reverse())

# 6. adjpitch
def adjpitch():
    while True:
        pitch_rate = float(input("Input pitch rate (-5 sampai 5):"))
        if not (-5 <= pitch_rate <= 5):
            print("Nilai must be between 1-5.")
        else:
            break
    shifted_audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=pitch_rate)
    export_librosa(shifted_audio)

# 7. adjspeed
def adjspeed():
    while True:
        nilai = float(input("Input nilai speed (1-5):"))
        if not (1 <= nilai <= 5):
            print("Nilai must be between 1-5.")
        else:
            break
    speed_audio = librosa.effects.time_stretch(audio, rate=nilai)
    export_librosa(speed_audio)

# 8. white noise
def white_noise():
    while True:
        noise_level = float(input("Input noise level (0.1 sampai 0.5):"))
        if not (0.1 <= noise_level <= 0.5):
            print("Nilai must be between 0.1 sampai 0.5.")
        else:
            break
    noise = np.random.normal(0, audio.std(), audio.size)
    export_librosa(audio + noise * noise_level)

# 9. random gain 
def random_gain():
    nilai = np.random.uniform(0.5, 1.5)
    print(f"Random gain x {nilai:.2f}")
    export_librosa(audio * nilai)

# 5. aug menu

# 6. main menu